<img align="right" src="https://panoptes-uploads.zooniverse.org/project_avatar/86c23ca7-bbaa-4e84-8d8a-876819551431.png" type="image/png" height=100 width=100>
</img>
<h1 align="left">Train & Evaluate a YOLO Model</h1>
<h4 align="left">SUBSIM Playday on EDITO · Written by the KSO Team</h4>

Welcome! In this notebook you will train a [YOLO](https://docs.ultralytics.com/) object-detection model on a small marine dataset and evaluate its performance — all in under 30 minutes.

The workflow is:

1. **Configure** – Point to the playday dataset (already downloaded for you)
2. **Verify** – Check that images, labels, and GPU are ready
3. **Train** – Run YOLO training on the dataset
4. **Evaluate** – Get publishable test-set metrics

Run cells **top to bottom**. After the guided run-through, feel free to go back and experiment — try a different model, change the number of epochs, or adjust the batch size and see how results change!

---

### Available Baseline Models from Ultralytics

| Family | Models (n / s / m / l / x) |
|--------|---------------------------|
| **YOLOv8** | `yolov8n.pt` &nbsp; `yolov8s.pt` &nbsp; `yolov8m.pt` &nbsp; `yolov8l.pt` &nbsp; `yolov8x.pt` |
| **YOLOv9** | `yolov9t.pt` &nbsp; `yolov9s.pt` &nbsp; `yolov9m.pt` &nbsp; `yolov9c.pt` &nbsp; `yolov9e.pt` |
| **YOLOv10** | `yolov10n.pt` &nbsp; `yolov10s.pt` &nbsp; `yolov10m.pt` &nbsp; `yolov10l.pt` &nbsp; `yolov10x.pt` |
| **YOLO11** | `yolo11n.pt` &nbsp; `yolo11s.pt` &nbsp; `yolo11m.pt` &nbsp; `yolo11l.pt` &nbsp; `yolo11x.pt` |

> **Tip:** To fine-tune from your own pretrained model, set `baseline_weights` to the path of your `.pt` file instead of an Ultralytics model name.

### Model Size Tips

- **Small datasets (~100–250 frames):** nano or small — larger models will overfit
- **Medium datasets (~250–750 frames):** medium for a good balance
- **Large datasets (750+ frames):** large or xlarge for best accuracy

Our playday dataset has ~100 images, so **nano** is the right choice to start.

---
## Phase 1: Configuration

The paths below are pre-filled for the playday dataset. Just run this cell.

> **Want to experiment later?** Come back here and change `baseline_weights` (try `yolo11s.pt`), raise `epochs`, or adjust `batch_size`.

In [ ]:
from pathlib import Path

# ── Paths ──
# Point to the playday dataset folder (the one containing data.yaml)
data_path = Path("")  # <-- e.g. "SUBSIM_playday_data/<dataset_name>"

# Where to save training runs
EXPERIMENT_ROOT = Path("models")

# ── Experiment ──
exp_name = ""  # <-- Name for this run, e.g. "my_first_model"
baseline_weights = ""  # <-- e.g. "yolo11n.pt" (nano, good for ~100 images)

# ── Training ──
epochs = (
    20  # How many passes over the dataset. Try 40-50 for better results if time allows
)
batch_size = 8
img_size = 640  # Standard YOLO input size

# ── Evaluation ──
conf_thres = 0.5  # Confidence threshold for test evaluation

# ── Derived (no need to edit) ──
data_yaml = data_path / "data.yaml"

print(f"Dataset:  {data_path}")
print(f"Output:   {EXPERIMENT_ROOT / exp_name}")
print(f"Model:    {baseline_weights}")
print(f"Epochs:   {epochs}")

---
## Phase 2: Verify Setup

Checks that paths exist and the environment is ready.

In [ ]:
import torch

# Check dataset
assert data_path.exists(), f"Dataset folder not found: {data_path}"
assert data_yaml.exists(), f"data.yaml not found: {data_yaml}"
print(f"Dataset:     {data_path}")

# Check for expected splits
for split in ("train", "valid"):
    split_dir = data_path / split
    assert split_dir.exists(), f"Required split missing: {split_dir}"
    n_images = (
        len(list((split_dir / "images").glob("*")))
        if (split_dir / "images").exists()
        else 0
    )
    print(f"  {split + ':':10s} {n_images} images")

# Check test split (optional but needed for Phase 4)
test_dir = data_path / "test"
if test_dir.exists():
    n_test = (
        len(list((test_dir / "images").glob("*")))
        if (test_dir / "images").exists()
        else 0
    )
    print(f"  {'test:':10s} {n_test} images")
else:
    print("  test:      NOT FOUND (Phase 4 will be skipped)")

# Check GPU
if torch.cuda.is_available():
    print(f"GPU:         {torch.cuda.get_device_name(0)}")
else:
    print("GPU:         None detected (training will be slower but still works)")

# Create output directory
EXPERIMENT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Output dir:  {EXPERIMENT_ROOT}")
print()
print("Ready to train!")

---
## Phase 3: Train Model

This trains YOLO on your dataset. With nano and 20 epochs on ~100 images, it should take just a few minutes.

Results are saved to `models/<exp_name>/`.

> **What to watch for:** The table printed each epoch shows `mAP50` (mean Average Precision at IoU 0.50). Higher is better — you should see it climbing as training progresses.

In [ ]:
from ultralytics import YOLO

print(f"Training {baseline_weights} for {epochs} epochs...")
print(f"Output: {EXPERIMENT_ROOT / exp_name}")
print("=" * 50 + "\n")

model = YOLO(baseline_weights)
results = model.train(
    data=str(data_yaml),
    epochs=epochs,
    batch=batch_size,
    imgsz=img_size,
    project=str(EXPERIMENT_ROOT),
    name=exp_name,
    exist_ok=True,
    plots=True,
)

train_dir = Path(results.save_dir)
best_weights = train_dir / "weights" / "best.pt"

print(f"\n{'=' * 50}")
print(f"Training complete!")
print(f"  Best weights: {best_weights}")

---
## Phase 4: Test Evaluation

Evaluates the best checkpoint on the held-out **test** split.
These are your publishable, unbiased metrics — the ones you would report in a paper or thesis.

> If the dataset has no `test/` split, this phase will be skipped.

In [ ]:
# Check that test split exists
test_dir = data_path / "test"
if not test_dir.exists():
    print("No test split found \u2014 skipping test evaluation.")
    print("To enable this, add a test/ folder to your dataset and re-run.")
else:
    best_weights = EXPERIMENT_ROOT / exp_name / "weights" / "best.pt"
    assert best_weights.exists(), f"Weights not found: {best_weights}"

    print(f"Model:  {best_weights}")
    print(f"Evaluating on test set (conf={conf_thres})...\n")

    model = YOLO(str(best_weights))
    test_results = model.val(
        data=str(data_yaml),
        split="test",
        conf=conf_thres,
        project=str(EXPERIMENT_ROOT / exp_name),
        name="test_eval",
        exist_ok=True,
        plots=True,
    )

    print(f"\n{'=' * 50}")
    print("TEST SET RESULTS")
    print("=" * 50)
    print(f"  Precision:  {test_results.box.mp:.3f}")
    print(f"  Recall:     {test_results.box.mr:.3f}")
    print(f"  mAP@50:     {test_results.box.map50:.3f}")
    print(f"  mAP@50-95:  {test_results.box.map:.3f}")
    print("=" * 50)
    print(f"\nResults saved to: {EXPERIMENT_ROOT / exp_name / 'test_eval'}")

---
## ✅ Done!

You just trained and evaluated a YOLO model on marine data. Here's what was created:

### Output Structure

```
models/<exp_name>/
    weights/
        best.pt              ← Best checkpoint — use this for inference
        last.pt              ← Final checkpoint
    test_eval/               ← Publishable results
        confusion_matrix.png
        PR_curve.png
        P_curve.png
        R_curve.png
        F1_curve.png
    results.png              ← Training curves (for monitoring only)
```
---

### Try It Yourself!

Go back to **Phase 1** and experiment:

- **Change the model:** Try `yolo11s.pt` (bigger) or `yolov8n.pt` (different architecture)
- **More epochs:** Set `epochs = 50` for better convergence
- **Different experiment name:** Change `exp_name` to compare runs side by side

---
### Saving Your Work

All files in this Jupyter instance will be deleted when you shut down the SUBSIM service.
To keep your trained model, copy it to your EDITO personal storage:

```python
!mc cp models/<exp_name>/weights/best.pt s3/$S3_BUCKET
```

See the [EDITO File Explorer](https://datalab.dive.edito.eu/file-explorer) to verify your saved files.

---
## 📖 Glossary

| Term | What it means | Why it matters |
|:-----|:-------------|:---------------|
| **Epoch** | One complete pass over the training dataset. Setting 50 epochs means the model "sees" each image and label exactly 50 times, learning what's in it. | More epochs give the model more chances to learn, but too many can lead to overfitting. |
| **Loss curves** | Plots showing how the model's training and validation error change over time. | If training loss improves but validation loss worsens, the model is memorising rather than learning; that's a sign to stop earlier or use a smaller model. |
| **Precision** | Of everything the model predicted as a detection, how many were actually correct? | High precision = few false alarms. A precision of 0.95 means 95% of detections are real. |
| **Recall** | Of all real objects present in the images, how many did the model find? | High recall = few misses. A recall of 0.80 means the model catches 80% of objects. |
| **mAP** | Mean Average Precision: a single summary score of detection quality across confidence thresholds. | The standard metric for comparing object detection models. Higher is better; reported as mAP@50 (lenient) and mAP@50-95 (strict). |
| **Overfitting** | When the model learns the training images too specifically and stops generalising to new data. | Common with small datasets and large models - this is why we recommend **nano** for ~100 images. |
| **Confidence threshold** | The minimum score a detection must have to be kept. | A threshold of 0.5 means "only show detections the model is ≥50% sure about". Lower catches more but adds noise. |
| **Batch size** | How many images the model processes at once during training. | Larger batches use more GPU memory. Reduce if you get out-of-memory errors. |